<a href="https://colab.research.google.com/github/kolshaan/Hackathon-2026-UpsideDown/blob/master/Zenith_Tiger_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q litellm openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 17.3 MB/s eta 0:00:00


## **Read Source Data**

In [11]:
import pandas as pd

url_demandforecast = 'https://raw.githubusercontent.com/kolshaan/Hackathon-2026-UpsideDown/refs/heads/master/demand_forecast.csv'
df_forecast = pd.read_csv(url_demandforecast)
url_inventory = 'https://raw.githubusercontent.com/kolshaan/Hackathon-2026-UpsideDown/refs/heads/master/current_inventory%20(1).csv'
df_inventory=pd.read_csv(url_inventory)

In [15]:
# df_inventory and df_forecast are already loaded
inventory_data = df_inventory.to_csv(index=False)
forecast_data = df_forecast.to_csv(index=False)

prompt = f"""
ROLE: You are 'The Watchdog', an Inventory Monitor Agent.
PURPOSE: Scans the warehouse network to identify where stock is critically low ("Distress") and where it is overflowing ("Excess").

DATA:
--- INVENTORY ---
{inventory_data}

--- DEMAND FORECAST ---
{forecast_data}
---

TASK:
1. Identify "Distress" SKUs (where On_Hand_Qty < Safety_Stock_Target or Forecast).
2. Identify "Excess" SKUs (where On_Hand_Qty > Safety_Stock_Target or Forecast).
3. Pair them up where a location with 'Excess' can supply a location with 'Needs'.

OUTPUT:
Return ONLY a JSON list of objects with this format:
[{{"SKU": "ID", "Needs": "Loc_A", "Has_Excess": "Loc_B"}}]
"""

## **Test API Gateway Connect to Gemini 2.0 Flash Model**

In [23]:

from openai import OpenAI

from google.colab import userdata
apikey = userdata.get('secret1')
base_url = "https://api.ai-gateway.tigeranalytics.com"

client = OpenAI(
    api_key=apikey,
    base_url=base_url
)

response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
        {"role": "user", "content": "Say hello from Gemini 2.0 Flash"}
    ]
)

print(response.choices[0].message.content)

Hello from Gemini 2.0 Flash! It's a pleasure to be communicating with you. How can I help you today?



## **The Watchdog**

In [19]:
response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
        {"role": "system", "content": "You are a supply chain analyst that only outputs valid JSON."},
        {"role": "user", "content": prompt}
    ],
    response_format={ "type": "json_object" } # This ensures Gemini returns valid JSON
)

In [20]:
watchdog_report = response.choices[0].message.content
print(watchdog_report)

[
  {
    "SKU": "ZEN-101",
    "Needs": "W01_New_Jersey",
    "Has_Excess": "W02_Chicago"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W01_New_Jersey"
  }
]


In [25]:
import json
import pandas as pd
from google.colab import data_table

# 1. Parse the JSON string into a Python list
report_data = json.loads(watchdog_report)

# 2. Convert to a DataFrame
report_df = pd.DataFrame(report_data)

# 3. Enable Colab's interactive table view
data_table.enable_dataframe_formatter()

# 4. Display the report
report_df

,SKU,Needs,Has_Excess
0,ZEN-101,W01_New_Jersey,W02_Chicago
1,ZEN-301,W07_Seattle,W01_New_Jersey


## **Test API Gateway Connect to Gemini 2.5 Flash Model**

In [22]:
import openai

from google.colab import userdata
apikey = userdata.get('secret1')  # Key generated from the ai-gateway.
base_url = "https://api.ai-gateway.tigeranalytics.com"
model = "gemini-2.5-flash"    # Model name present in the supported list.
user_query = "What is the capital of India?"   # The text we want to ask Model.

client = openai.OpenAI(api_key =  apikey,
                       base_url = base_url)
response = client.chat.completions.create(model=model,
                                          messages = [{"role": "user",
                                                       "content": user_query
                                                      }
                                                     ])
print(response.choices[0].message.content)

The capital of India is **New Delhi**.
